# 🎤 Isle Voice → your voice (free RVC on Google Colab)

This notebook turns the **draft vocal** from [Isle Voice](https://madinc47-prog.github.io/isle-voice/) into **your own voice**, using free open‑source [Applio](https://github.com/IAHispano/Applio) (RVC) on Colab's free GPU and **your** Google Drive.

**Just do this:** menu **Runtime → Run all** (on a phone/tablet: ⋮ menu → Runtime → Run all). Then:
1. **Allow Google Drive** when Google asks (your voice model is saved there so you only train once).
2. **Pick your Isle Voice zip** when the *Choose files* button appears (or put the zip in *Google Drive › IsleVoice › inbox* before you start).
3. **Wait.** First time only, it trains your voice: about **45–120 minutes** on the free T4 GPU, plus ~5–10 min setup. Next time it skips training and converts in ~5–10 min.
4. Your converted vocal **downloads automatically** (`…_my_voice.wav`) and a copy goes to *Google Drive › IsleVoice › out*. Import it back in Isle Voice.

Keep this tab open while it runs. If Colab disconnects, just **Run all** again: training continues from the last save in your Drive.
Your audio stays between your device, this Colab session and your own Google Drive. Only use your own voice or a voice you have permission to use. Applio is MIT-licensed; see its [terms of use](https://github.com/IAHispano/Applio/blob/main/TERMS_OF_USE.md).

In [ ]:
# @title Step 1 · Connect Google Drive and check the GPU
import os, sys, json, glob, shutil, subprocess, zipfile, time, threading, wave
from pathlib import Path

TEST = os.environ.get("ISLEVOICE_TEST")          # local dry run only; empty on Colab
ROOT = Path(TEST) if TEST else Path("/content")
DRIVE = ROOT / "drive" / "MyDrive" / "IsleVoice"
APPLIO = ROOT / "Applio"
PY = str(APPLIO / ".venv" / "bin" / "python")
APPLIO_COMMIT = "d19f1efec0f0acf30a6e32fd0b96548df11a3816"                     # pinned, so the commands below keep working

def sh(cmd, cwd=None):
    """Run a command, stream its output, stop the notebook with a clear message if it fails."""
    shown = cmd if isinstance(cmd, str) else " ".join(map(str, cmd))
    print("\n$", shown[:300], flush=True)
    p = subprocess.Popen(cmd, cwd=cwd, shell=isinstance(cmd, str), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env=dict(os.environ, PATH=os.path.expanduser("~/.local/bin") + ":" + os.environ.get("PATH", "")))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"This step failed (exit {p.returncode}). Scroll up for the reason, then Runtime → Run all to try again.")

def stop(msg):
    print("\n" + "=" * 60 + "\n" + msg + "\n" + "=" * 60)
    raise SystemExit(msg)

IN_COLAB = not TEST
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
for d in ("inbox", "models", "out"):
    (DRIVE / d).mkdir(parents=True, exist_ok=True)
print("✅ Google Drive connected:", DRIVE)

if IN_COLAB:
    gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True)
    if gpu.returncode != 0 or not gpu.stdout.strip():
        stop("❌ No GPU. Menu Runtime → Change runtime type → T4 GPU → Save, then Runtime → Run all.\n"
             "If Colab says no GPU is available, the free GPUs are busy or your daily free time is used up: try again in a few hours.")
    print("✅ GPU:", gpu.stdout.strip())

In [ ]:
# @title Step 2 · Install Applio (about 5–10 minutes, every new session)
marker = APPLIO / ".isle_voice_ready"
if marker.exists():
    print("✅ Applio already installed in this session.")
else:
    if not (APPLIO / ".git").exists():
        sh(["git", "clone", "-q", "https://github.com/IAHispano/Applio", str(APPLIO)])
    sh(["git", "-c", "advice.detachedHead=false", "checkout", "-q", APPLIO_COMMIT], cwd=str(APPLIO))
    sh("apt-get -qq update -y && apt-get -qq install -y portaudio19-dev > /dev/null")
    sh("curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null")
    sh("uv venv .venv --python 3.12 --seed -q", cwd=str(APPLIO))
    sh(f"uv pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match --python {PY}", cwd=str(APPLIO))
    sh(f"uv pip install -q matplotlib-inline websockets --python {PY}", cwd=str(APPLIO))
    sh([PY, "core.py", "prerequisites", "--models", "--pretraineds-hifigan", "--exe"], cwd=str(APPLIO))
    if not (APPLIO / "assets" / "config.json").exists():
        shutil.copy(APPLIO / "assets" / "config_template.json", APPLIO / "assets" / "config.json")
    marker.write_text("ok")
    print("✅ Applio installed.")

In [ ]:
# @title Step 3 · Pick your Isle Voice zip
def newest_zip(folder):
    zips = sorted(Path(folder).glob("*.zip"), key=lambda p: p.stat().st_mtime, reverse=True)
    return zips[0] if zips else None

def open_job(zpath, dest):
    dest = Path(dest)
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True)
    with zipfile.ZipFile(zpath) as z:
        for m in z.namelist():                      # refuse odd paths inside the zip
            if m.startswith("/") or ".." in Path(m).parts:
                raise ValueError("Unexpected file in zip: " + m)
        z.extractall(dest)
    jf = dest / "job.json"
    if not jf.exists() or not (dest / "draft_vocal.wav").exists():
        stop("❌ That zip isn't an Isle Voice zip (no job.json / draft_vocal.wav). In Isle Voice tap Apply my voice to make one.")
    job = json.loads(jf.read_text())
    job["model_name"] = "".join(c if c.isalnum() or c in "_-" else "_" for c in str(job.get("model_name") or "my_voice"))[:40] or "my_voice"
    job["output_name"] = Path(str(job.get("output_name") or "song_my_voice.wav")).name
    return job

JOB_DIR = ROOT / "job"
zpath = newest_zip(DRIVE / "inbox")
age_h = (time.time() - zpath.stat().st_mtime) / 3600 if zpath else None
if zpath and age_h < 6:
    print(f"📦 Using {zpath.name} from Google Drive › IsleVoice › inbox")
elif IN_COLAB:
    from google.colab import files
    print("👇 Tap Choose files and pick the …_isle_voice.zip from your Downloads.")
    up = files.upload()
    if not up:
        stop("❌ No file picked. Runtime → Run all to try again.")
    name = next(iter(up))
    zpath = ROOT / name
    zpath.write_bytes(up[name])
elif not zpath:
    stop("❌ No zip found.")
job = open_job(zpath, JOB_DIR)
bank = sorted((JOB_DIR / "voice_bank").glob("*.wav"))
def wav_seconds(p):
    try:
        with wave.open(str(p)) as w:
            return w.getnframes() / float(w.getframerate())
    except Exception:
        return 0.0
bank_secs = sum(wav_seconds(p) for p in bank)
draft_secs = wav_seconds(JOB_DIR / "draft_vocal.wav")
print(f"🎵 Song: {job.get('song')} · draft vocal {draft_secs:.0f}s · voice bank {len(bank)} clips / {bank_secs/60:.1f} min · model '{job['model_name']}'")

In [ ]:
# @title Step 4 · Train your voice (first time only — about 45–120 min)
MODEL = job["model_name"]
MODEL_DIR = DRIVE / "models" / MODEL               # lives in YOUR Google Drive, reused next time
CKPT_DIR = MODEL_DIR / "training"                  # in-progress checkpoints, so a disconnect can resume
LOGS = APPLIO / "logs" / MODEL

def find_model(d):
    d = Path(d)
    pths = sorted([p for p in d.glob(f"{MODEL}*.pth") if not p.name.startswith(("G_", "D_"))], key=lambda p: p.stat().st_mtime, reverse=True)
    idx = sorted(d.glob("*.index"), key=lambda p: p.stat().st_mtime, reverse=True)
    return (pths[0], idx[0]) if pths and idx else None

def backup(src, dst, patterns):
    dst.mkdir(parents=True, exist_ok=True)
    for pat in patterns:
        for f in Path(src).glob(pat):
            t = dst / f.name
            if not t.exists() or t.stat().st_mtime < f.stat().st_mtime or t.stat().st_size != f.stat().st_size:
                shutil.copy2(f, t)

def epochs_for(minutes):
    return 300 if minutes <= 5 else 200 if minutes <= 15 else 150

found = find_model(MODEL_DIR)
if found:
    print(f"✅ Your voice model is already trained ({found[0].name}). Skipping training.")
else:
    if not bank:
        stop("❌ No trained model in Google Drive › IsleVoice › models › " + MODEL + " and this zip has no voice bank.\n"
             "In Isle Voice: untick “My voice model is already trained”, record your voice bank (10–30 min), tap Apply my voice again.")
    if bank_secs < 60:
        print("⚠️ Less than a minute of voice: the model will sound rough. 10–30 minutes is best.")
    epochs = epochs_for(bank_secs / 60)
    print(f"🏋️ Training '{MODEL}' on {bank_secs/60:.1f} min of your voice for {epochs} epochs. First time only. Keep this tab open.")
    DATASET = ROOT / "dataset" / MODEL
    if DATASET.exists():
        shutil.rmtree(DATASET)
    DATASET.mkdir(parents=True)
    for p in bank:
        shutil.copy2(p, DATASET / p.name)
    LOGS.mkdir(parents=True, exist_ok=True)
    if CKPT_DIR.exists():                           # resume after a disconnect
        backup(CKPT_DIR, LOGS, ["G_*.pth", "D_*.pth", "*.json"])
        print("↩️ Found saved training in your Drive — continuing from there.")
    sh([PY, "core.py", "preprocess", "--model-name", MODEL, "--dataset-path", str(DATASET), "--sample-rate", "40000",
        "--cpu-cores", str(os.cpu_count() or 2), "--cut-preprocess", "Automatic", "--chunk-len", "3.0", "--overlap-len", "0.3",
        "--normalization-mode", "none", "--noise-reduction-strength", "0.7"], cwd=str(APPLIO))
    sh([PY, "core.py", "extract", "--model-name", MODEL, "--f0-method", "rmvpe", "--sample-rate", "40000",
        "--cpu-cores", str(os.cpu_count() or 2), "--gpu", "0", "--embedder-model", "contentvec", "--include-mutes", "2"], cwd=str(APPLIO))
    done = threading.Event()
    def saver():                                    # copy checkpoints to Drive every 5 minutes while training
        while not done.wait(300):
            try:
                backup(LOGS, CKPT_DIR, ["G_*.pth", "D_*.pth", "*.json"])
            except Exception as e:
                print("(backup skipped:", e, ")")
    threading.Thread(target=saver, daemon=True).start()
    try:
        sh([PY, "core.py", "train", "--model-name", MODEL, "--save-every-epoch", "10", "--total-epoch", str(epochs),
            "--sample-rate", "40000", "--batch-size", "8", "--gpu", "0", "--vocoder", "HiFi-GAN", "--pretrained",
            "--save-only-latest"], cwd=str(APPLIO))
    finally:
        done.set()
        backup(LOGS, CKPT_DIR, ["G_*.pth", "D_*.pth", "*.json"])
    if not list(LOGS.glob("*.index")):
        sh([PY, "core.py", "index", "--model-name", MODEL, "--index-algorithm", "Auto"], cwd=str(APPLIO))
    final = find_model(LOGS)
    if not final:
        stop("❌ Training finished but no model file was found. Scroll up for errors, then Runtime → Run all (it resumes).")
    for f in final:
        shutil.copy2(f, MODEL_DIR / f.name)
    shutil.rmtree(CKPT_DIR, ignore_errors=True)      # finished: keep only the small final model
    found = find_model(MODEL_DIR)
    print(f"✅ Trained and saved to Google Drive › IsleVoice › models › {MODEL} ({found[0].name}). Next time training is skipped.")

In [ ]:
# @title Step 5 · Convert the draft into your voice and download it
pth, index = found
OUT = ROOT / "out"
OUT.mkdir(exist_ok=True)
out_wav = OUT / job["output_name"]
if out_wav.exists():
    out_wav.unlink()
pitch = max(-12, min(12, int(job.get("pitch") or 0)))
cmd = [PY, "core.py", "infer", "--input-path", str(JOB_DIR / "draft_vocal.wav"), "--output-path", str(out_wav),
       "--pth-path", str(pth), "--index-path", str(index), "--pitch", str(pitch), "--f0-method", "rmvpe",
       "--index-rate", "0.6", "--volume-envelope", "0.6", "--protect", "0.4", "--export-format", "WAV",
       "--embedder-model", "contentvec"]
if draft_secs > 45:
    cmd.append("--split-audio")
sh(cmd, cwd=str(APPLIO))
if not out_wav.exists():
    stop("❌ Conversion didn't produce a file. Scroll up for the error, then Runtime → Run all.")
shutil.copy2(out_wav, DRIVE / "out" / out_wav.name)
print(f"✅ Done! {out_wav.name} is downloading now, and a copy is in Google Drive › IsleVoice › out.")
print("➡️ Go back to Isle Voice → Make my song → Import my voice, and pick this file.")
if IN_COLAB:
    from IPython.display import Audio, display
    display(Audio(str(out_wav)))
    from google.colab import files
    files.download(str(out_wav))